# Signed Jumps 04 — jumps at 1-second resolution

Protocol: `discussions/Signed_Jumps/007_claude_sj04_preregistration.md` (fixed before this run).

SJ 01 jump minutes (engine unchanged) → onset second from spot 1s klines → **P1** early activity build-up, **P2** split events (bar-boundary artefact), **P3** continuation from onset+5 s to onset+65 s.

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj, signed_jumps_seconds as ss
for m in (sj,ss):importlib.reload(m)
assert sj.VERSION.startswith('SJ01-0.1') and ss.VERSION=='SJ04-0.1.0'
SYNTHETIC=os.getenv('SJ04_SYNTHETIC','0')=='1'
p=sj.Params();sp=ss.SecParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj04-');OUT=Path(tmp.name)/'04'
    p=sj.Params(n_boot=200,half_lives=(1,));sp=ss.SecParams(n_boot=300)
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'04'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps_seconds.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(ss.registration(sp),indent=1))

.....                                                                    [100%]
5 passed in 7.31s


{
 "baseline_s": 3600,
 "profile": [
  -600,
  300
 ],
 "p1_window": [
  -120,
  -31
 ],
 "split_gap_s": 30,
 "split_share_threshold": 0.5,
 "p3_delta_s": 5,
 "p3_h_s": 60,
 "grid_delta": [
  1,
  5,
  10,
  30
 ],
 "grid_h": [
  10,
  30,
  60,
  300
 ],
 "placebo_per_event": 3,
 "placebo_exclusion_min": 10,
 "cluster_gap_min": 5,
 "costs_bp": [
  12.0,
  22.0
 ],
 "n_boot": 2000,
 "status": "PREREGISTERED_SJ007",
 "version": "SJ04-0.1.0",
 "gates": {
  "P1": "event-minus-placebo activity over [onset-120, onset-31] s: q2.5 > 0",
  "P2": "share of consecutive same-sign jump minutes whose peak seconds are <= 30 s apart; > 0.5 => artefact",
  "P3": "signed spot return onset+5 s -> onset+65 s: gross q2.5 > 0"
 }
}


## SJ 01 jumps (engine unchanged) and 1-second data

In [2]:
T0=time.time()
if SYNTHETIC:
    secs,one=ss.simulate_seconds(n_days=14,n_assets=len(p.assets),seed=3,precursor=0.5,drift_bp=5,straddle_share=0.3)
    secs={a:v for a,v in zip(p.assets,secs.values())};one={a:v for a,v in zip(p.assets,one.values())}
    assets,pan,_=sj.prepare(one,p)
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={A:sj.load_spot_1m(BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet') for A in p.assets}
    assets,pan,_=sj.prepare(S,p)
    q01=pd.read_csv(ROOT/'notebooks'/'Signed_Jumps'/'01'/'01_data_quality.csv').set_index('asset')
    for i,a in enumerate(assets):
        m=pan.asset_idx==i
        for s in sj.SIGNS:
            for per,mm in (('dev',pan.dev),('test',~pan.dev)):
                assert int(pan.J[s][m&mm].sum())==int(q01.loc[a.name,f'{s}_{per}']),(a.name,s,per)
    print('jump counts reproduce SJ 01')
    secs={}
    for A in p.assets:
        files=sorted((BIN/'spot_1s'/f'{A}USDT').glob(f'{A}USDT_spot_1s_*.parquet'))
        assert files,f'no 1s files for {A}'
        secs[A]=ss.load_spot_1s(files);print(A,len(files),'files',secs[A].index.min(),'->',secs[A].index.max(),f'{len(secs[A]):,} s')
ev=ss.jump_minutes(assets,pan,sp.cluster_gap_min)
keep=[]
for a,g in ev.groupby('asset'):
    lo,hi=secs[a].index.min()+pd.Timedelta(seconds=sp.baseline_s-sp.profile[0]),secs[a].index.max()-pd.Timedelta(seconds=sp.profile[1]+400)
    keep.append(g[(g['T']>=lo)&(g['T']<=hi)])
ev=pd.concat(keep).sort_values(['asset','T']).reset_index(drop=True)
ev=ss.onset_seconds(ev,secs)
print(f'{len(ev)} jump minutes in 1s window ({ev.starter.sum()} cluster starters); onset found for {ev.onset.notna().sum()} | {time.time()-T0:.0f}s')
desc=ev.groupby('asset').agg(jumps=('T','size'),starters=('starter','sum'),onset_found=('onset','count'),
     median_onset_u=('onset_u','median'),median_peak_share=('peak_share','median'),mean_abs_minute_bp=('minute_bp',lambda x:x.abs().mean()))
display(desc.round(2));desc.to_csv(OUT/'04_events_summary.csv')
hist=pd.cut(ev.onset_u,[0,10,20,30,40,50,60]).value_counts(normalize=True).sort_index()
print('onset second within the jump minute:');display(hist.round(3))
ev.to_csv(OUT/'04_events.csv.gz',index=False)

D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:172: RuntimeWarning: divide by zero encountered in log
  feats.append(np.log(m))
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: divide by zero encountered in divide
  Z = z / np.sqrt(sig2c)
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: invalid value encountered in divide
  Z = z / np.sqrt(sig2c)


jump counts reproduce SJ 01
BTC 3 files 2026-07-01 00:00:01+00:00 -> 2026-10-01 00:00:00+00:00 7,948,800 s
ETH 3 files 2026-07-01 00:00:01+00:00 -> 2026-10-01 00:00:00+00:00 7,948,800 s
SOL 3 files 2026-07-01 00:00:01+00:00 -> 2026-10-01 00:00:00+00:00 7,948,800 s
XRP 3 files 2026-07-01 00:00:01+00:00 -> 2026-10-01 00:00:00+00:00 7,948,800 s
2186 jump minutes in 1s window (1868 cluster starters); onset found for 2186 | 100s


,jumps,starters,onset_found,median_onset_u,median_peak_share,mean_abs_minute_bp
asset,,,,,,
BTC,651,563,651,29.0,0.33,21.29
ETH,563,474,563,29.0,0.31,36.29
SOL,471,399,471,28.0,0.30,40.78
XRP,501,432,501,27.0,0.31,39.00


onset second within the jump minute:


onset_u
(0, 10]     0.129
(10, 20]    0.205
(20, 30]    0.215
(30, 40]    0.212
(40, 50]    0.163
(50, 60]    0.076
Name: proportion, dtype: float64

## P1 — activity build-up before onset

In [3]:
act={a:ss.activity(secs[a],sp.baseline_s) for a in secs}
plc=ss.placebo_onsets(ev,secs,sp)
r1=ss.p1_test(ev,plc,secs,act,sp);print(json.dumps(r1,indent=1))
pr=ss.profile(ev,plc,secs,act,sp,bin_s=10);pr.to_csv(OUT/'04_profile_10s.csv',index=False)
pr1=ss.profile(ev,plc,secs,act,sp,bin_s=1);pr1.to_csv(OUT/'04_profile_1s.csv',index=False)
display(pr.set_index('bin_start_s').loc[[-600,-300,-180,-120,-90,-60,-40,-30,-20,-10,0,10,30,60,120,290]].round(3))

{
 "event_mean": -0.9018953718196534,
 "placebo_mean": -1.2114260389425777,
 "excess": 0.3095306671229243,
 "q025": 0.25382886437556756,
 "q975": 0.36915375950815615,
 "n_events": 1868,
 "n_placebo": 6558,
 "status": "PASS"
}


who,event,placebo,excess
bin_start_s,,,
-600,-1.104,-1.199,0.095
-300,-1.020,-1.205,0.185
-180,-1.028,-1.199,0.171
-120,-0.984,-1.200,0.216
-90,-0.903,-1.213,0.310
-60,-0.867,-1.209,0.341
-40,-0.814,-1.215,0.401
-30,-0.715,-1.204,0.489
-20,-0.522,-1.216,0.694


## P2 — split events (bar-boundary artefact?)

In [4]:
pairs,r2=ss.split_pairs(ev,sp);print(json.dumps(r2,indent=1))
pairs.to_csv(OUT/'04_pairs.csv',index=False)
if len(pairs):display(pairs.groupby('same_sign').gap_s.describe().round(1))

{
 "pairs_same_sign": 104,
 "pairs_opposite": 55,
 "split_share_same_sign": 0.15384615384615385,
 "split_share_opposite": 0.4,
 "median_gap_same_sign_s": 51.5,
 "verdict": "MOSTLY_DISTINCT_EVENTS"
}


,count,mean,std,min,25%,50%,75%,max
same_sign,,,,,,,,
False,55.0,39.0,22.9,1.0,18.5,40.0,58.5,99.0
True,104.0,53.9,23.2,9.0,36.0,51.5,71.0,107.0


## P3 — continuation after onset (seconds)

In [5]:
e3=ss.post_onset(ev,secs,sp)
t3,r3=ss.p3_table(e3,sp);print(json.dumps(r3,indent=1,default=float))
t3.to_csv(OUT/'04_post_onset_grid.csv',index=False)
display(t3.pivot_table(index='delta_s',columns='h_s',values='gross_bp').round(2))
display(t3.pivot_table(index='delta_s',columns='h_s',values='share_positive').round(3))

{
 "gross_bp": 8.174507842914316,
 "q025": 6.694245440394582,
 "q975": 9.749123488537945,
 "n": 1868,
 "status": "PASS",
 "net_12bp": -3.8254921570856837,
 "net_22bp": -13.825492157085684
}


h_s,10,30,60,300
delta_s,,,,
1,5.53,9.11,10.49,11.27
5,4.28,7.05,8.17,8.88
10,2.56,4.85,5.33,6.36
30,0.96,1.52,1.10,2.46


h_s,10,30,60,300
delta_s,,,,
1,0.766,0.854,0.814,0.637
5,0.720,0.783,0.729,0.581
10,0.650,0.694,0.640,0.542
30,0.491,0.506,0.482,0.455


## Decision

In [6]:
dec={'version':ss.VERSION,'status':sp.status,'P1_early_buildup':r1,'P2_split_events':r2,'P3_continuation':r3,
     'events':int(len(ev)),'starters':int(ev.starter.sum()),'synthetic':SYNTHETIC}
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'04_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'04_manifest.json').write_text(json.dumps({'registration':ss.registration(sp),'runtime_s':time.time()-T0},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')

{
 "version": "SJ04-0.1.0",
 "status": "PREREGISTERED_SJ007",
 "P1_early_buildup": {
  "event_mean": -0.9018953718196534,
  "placebo_mean": -1.2114260389425777,
  "excess": 0.3095306671229243,
  "q025": 0.25382886437556756,
  "q975": 0.36915375950815615,
  "n_events": 1868,
  "n_placebo": 6558,
  "status": "PASS"
 },
 "P2_split_events": {
  "pairs_same_sign": 104,
  "pairs_opposite": 55,
  "split_share_same_sign": 0.15384615384615385,
  "split_share_opposite": 0.4,
  "median_gap_same_sign_s": 51.5,
  "verdict": "MOSTLY_DISTINCT_EVENTS"
 },
 "P3_continuation": {
  "gross_bp": 8.174507842914316,
  "q025": 6.694245440394582,
  "q975": 9.749123488537945,
  "n": 1868,
  "status": "PASS",
  "net_12bp": -3.8254921570856837,
  "net_22bp": -13.825492157085684
 },
 "events": 2186,
 "starters": 1868,
 "synthetic": false
}
total 130s
